# DFS Weekly Pipeline

> **DEV — never validated against a live DraftKings slate.** The 2025 season came and
> went without an end-to-end live test, and the only salary file in the repo is still
> `dk_salaries_2025_week10_synthetic.csv`. Treat all outputs as experimental until a
> real slate has been run end-to-end and logged.

**Run this notebook each week** to generate an optimised DraftKings NFL Classic lineup.

### Workflow
1. Run `predict_fantasy.ipynb` for the target week (or let GitHub Actions do it).
2. When the DK slate posts, export the salary CSV from any NFL Classic contest lobby
   (*Available Players → Export to CSV*) and note its path.
3. Set `CSV_PATH` (and optionally `SEASON` / `WEEK`) in the Parameters cell below.
4. Run all cells top-to-bottom. Review the player pool, adjust `LOCKED` / `EXCLUDED`,
   and re-run the optimizer cell as needed.

### DraftKings NFL Classic roster
| Slot | Count |
|---|---|
| QB | 1 |
| RB | 2 |
| WR | 3 |
| TE | 1 |
| FLEX (RB/WR/TE) | 1 |
| DST | 1 |
| **Total** | **9** |

Salary cap: **$50,000**. Max 8 players from the same team.


## Parameters

Set `CSV_PATH` to the DK salary export for this week's slate.
`SEASON` and `WEEK` default to `None`, which auto-detects the most recent projection
file in `fantasy/fantasy_projections/` — override them if you want a specific week.

`LOCKED` / `EXCLUDED` accept player names exactly as they appear in the DK salary CSV.


In [11]:
# Parameters (papermill-injectable)
# Path to the DK salary CSV exported from the contest lobby
CSV_PATH = "dk_salaries.csv"
# None = auto-detect from the most recent projection file
SEASON = None
WEEK = None
BUDGET = 50_000
LOCKED = []
EXCLUDED = []


## Setup

All optimizer logic is defined here so this notebook is fully self-contained.
No import from a separate module is required.


In [ ]:
import io
import re
import math
import difflib
from pathlib import Path

import pandas as pd
import pulp

# ── Path resolution ────────────────────────────────────────────────────────
_HERE = Path().resolve()
PROJ_DIR = next(
    (p for p in [
        _HERE / "fantasy_projections",
        _HERE / "fantasy" / "fantasy_projections",
        _HERE.parent / "fantasy_projections",
        _HERE.parent / "fantasy" / "fantasy_projections",
        _HERE.parent.parent / "fantasy" / "fantasy_projections",
    ] if p.exists()),
    _HERE / "fantasy_projections",
)
if not PROJ_DIR.exists():
    raise RuntimeError(f"Could not find fantasy_projections directory. PROJ_DIR={PROJ_DIR}")

_DK_COL_MAP = {
    "Position":         "position",
    "Name":             "name",
    "Salary":           "salary",
    "TeamAbbrev":       "team",
    "AvgPointsPerGame": "avg_pts",
    "Game Info":        "game_info",
}

# Per-stat projection columns pulled through from proj file for DK scoring
STAT_COLS = [
    "pred_qb_pass_yards", "pred_qb_rush_yards",
    "pred_rush_yards",    "pred_rec_yards",
    "pred_wr_receptions", "pred_wr_rec_yards",
    "pred_te_receptions", "pred_te_rec_yards",
]

def _norm(name):
    name = name.lower().strip()
    name = re.sub(r"[''\-\.,]", "", name)
    name = re.sub(r"\s+(jr|sr|ii|iii|iv|v)$", "", name)
    return re.sub(r"\s+", " ", name)

def available_weeks():
    files = sorted(PROJ_DIR.glob("projections_*_week*.csv"), reverse=True)
    out = []
    for f in files:
        m = re.match(r"projections_(\d{4})_week(\d{2})\.csv", f.name)
        if m:
            out.append((int(m.group(1)), int(m.group(2))))
    return out

def load_projections(season, week):
    path = PROJ_DIR / f"projections_{season}_week{week:02d}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No projection file: {path}")
    df = pd.read_csv(path)
    df["_norm"] = df["player_display_name"].apply(_norm)
    return df

def load_dk_salaries(path_or_bytes):
    raw = open(path_or_bytes, "rb").read() if isinstance(path_or_bytes, (str, Path)) else path_or_bytes
    df  = pd.read_csv(io.BytesIO(raw))
    df  = df.rename(columns={k: v for k, v in _DK_COL_MAP.items() if k in df.columns})
    df["salary"]  = pd.to_numeric(df["salary"].astype(str).str.replace("$","",regex=False).str.replace(",","",regex=False), errors="coerce")
    df["avg_pts"] = pd.to_numeric(df.get("avg_pts", 0), errors="coerce").fillna(0)
    df = df.dropna(subset=["salary", "name", "position"])
    df["_norm"] = df["name"].apply(_norm)
    keep = ["position","name","salary","team","avg_pts","game_info","_norm"]
    return df[[c for c in keep if c in df.columns]].copy()


def merge_projections(dk_df, proj_df):
    proj_norms  = proj_df["_norm"].tolist()
    proj_lookup = {r["_norm"]: r for _, r in proj_df.iterrows()}
    stat_cols   = [c for c in STAT_COLS if c in proj_df.columns]

    pts_list, match_list = [], []
    stat_rows = {c: [] for c in stat_cols}

    for _, row in dk_df.iterrows():
        if row["position"] == "DST":
            pts_list.append(row.get("avg_pts", 0))
            match_list.append("dst")
            for c in stat_cols: stat_rows[c].append(None)
            continue
        hits = difflib.get_close_matches(row["_norm"], proj_norms, n=1, cutoff=0.72)
        if hits:
            proj_row = proj_lookup[hits[0]]
            pts_list.append(proj_row["projected_pts"])
            match_list.append("model")
            for c in stat_cols:
                stat_rows[c].append(proj_row.get(c))
        else:
            pts_list.append(row.get("avg_pts", 0))
            match_list.append("dk_avg")
            for c in stat_cols: stat_rows[c].append(None)

    result = dk_df.drop(columns=["_norm"], errors="ignore").copy()
    result["proj_pts"] = pts_list
    result["match"]    = match_list
    for c, vals in stat_rows.items():
        result[c] = vals
    result["value"] = (result["proj_pts"] / (result["salary"] / 1000)).round(2)
    return result.reset_index(drop=True)

def _norm_sf(threshold, mu, sigma):
    """P(X >= threshold) for X ~ Normal(mu, sigma) using stdlib math only."""
    return 0.5 * math.erfc((threshold - mu) / (sigma * math.sqrt(2)))

def calc_dk_proj_pts(players):
    """Convert half-PPR projected_pts to DraftKings Classic full-PPR + milestone bonuses.

    DK Classic differences vs half-PPR:
      Reception       : +0.5 pts/rec  (DK full-PPR=1.0, our model=0.5 half-PPR)
      300+ pass yards : +3 pts bonus (QB)
      100+ rush yards : +3 pts bonus (RB)
      100+ rec yards  : +3 pts bonus (WR, TE)

    Milestone bonuses weighted by P(reaching milestone) via Normal approximation
    with position-typical per-game std devs (QB pass sigma=65, RB rush sigma=37,
    WR rec sigma=28, TE rec sigma=22).
    """
    dfs_pts = []
    for _, row in players.iterrows():
        pts = float(row["proj_pts"])
        pos = row["position"]

        if pos == "QB":
            mu_pass = float(row.get("pred_qb_pass_yards") or 0)
            if mu_pass > 0:
                pts += _norm_sf(300, mu_pass, 65) * 3

        elif pos == "RB":
            mu_rush = float(row.get("pred_rush_yards") or 0)
            mu_rec  = float(row.get("pred_rec_yards")  or 0)
            if mu_rush > 0:
                pts += _norm_sf(100, mu_rush, 37) * 3
            if mu_rec > 0:
                pts += 0.5 * (mu_rec / 7.0)          # half-PPR -> full PPR (avg ~7 yds/rec for RBs)

        elif pos == "WR":
            recs   = float(row.get("pred_wr_receptions") or 0)
            mu_rec = float(row.get("pred_wr_rec_yards")  or 0)
            pts += 0.5 * recs
            if mu_rec > 0:
                pts += _norm_sf(100, mu_rec, 28) * 3

        elif pos == "TE":
            recs   = float(row.get("pred_te_receptions") or 0)
            mu_rec = float(row.get("pred_te_rec_yards")  or 0)
            pts += 0.5 * recs
            if mu_rec > 0:
                pts += _norm_sf(100, mu_rec, 22) * 3

        dfs_pts.append(round(pts, 2))
    return dfs_pts

def _assign_slots(lineup_df):
    df = lineup_df.copy().reset_index(drop=True)
    slots = [""] * len(df)
    seen = {"RB":0,"WR":0,"TE":0}
    limits = {"RB":2,"WR":3,"TE":1}
    for i, row in df.iterrows():
        pos = row["position"]
        if pos in ("QB","DST"):
            slots[i] = pos
        elif pos in seen:
            seen[pos] += 1
            slots[i] = pos if seen[pos] <= limits[pos] else "FLEX"
    df.insert(0, "Slot", slots)
    return df

def optimize_lineup(players, budget=50_000, locked=None, excluded=None):
    df = players.reset_index(drop=True)
    locked = set(locked or []); excluded = set(excluded or [])
    n = len(df)
    prob = pulp.LpProblem("DFS", pulp.LpMaximize)
    x = [pulp.LpVariable(f"x{i}", cat="Binary") for i in range(n)]
    def pidx(pos): return [i for i in range(n) if df.iloc[i]["position"] == pos]
    qbs, rbs, wrs, tes, dsts = (pidx(p) for p in ["QB","RB","WR","TE","DST"])
    prob += pulp.lpSum(df.iloc[i]["dfs_proj_pts"] * x[i] for i in range(n))
    prob += pulp.lpSum(df.iloc[i]["salary"]       * x[i] for i in range(n)) <= budget
    prob += pulp.lpSum(x[i] for i in range(n)) == 9
    prob += pulp.lpSum(x[i] for i in qbs)  == 1
    prob += pulp.lpSum(x[i] for i in dsts) == 1
    prob += pulp.lpSum(x[i] for i in rbs)  >= 2
    prob += pulp.lpSum(x[i] for i in wrs)  >= 3
    prob += pulp.lpSum(x[i] for i in tes)  >= 1
    for team in df["team"].dropna().unique():
        tidx = [i for i in range(n) if df.iloc[i]["team"] == team]
        if len(tidx) > 8: prob += pulp.lpSum(x[i] for i in tidx) <= 8
    for name in locked:
        for i in df[df["name"] == name].index: prob += x[i] == 1
    for name in excluded:
        for i in df[df["name"] == name].index: prob += x[i] == 0
    status = prob.solve(pulp.PULP_CBC_CMD(msg=0))
    if pulp.LpStatus[status] != "Optimal": return None
    selected = [i for i in range(n) if pulp.value(x[i]) > 0.5]
    order = {"QB":0,"RB":1,"WR":2,"TE":3,"DST":5}
    lineup = df.iloc[selected].copy()
    lineup["_sort"] = lineup["position"].map(order).fillna(4)
    lineup = lineup.sort_values("_sort").drop(columns=["_sort"])
    return _assign_slots(lineup)

print("All functions loaded.")
print(f"Projections directory: {PROJ_DIR}  (exists: {PROJ_DIR.exists()})")


## Step 1 — Load Model Projections

Our XGBoost models produce `projected_pts` (half-PPR) for each skill-position player.
We use these rather than DK's season average because they incorporate:
- Current injury status and practice participation
- Depth chart position (starter vs. backup)
- Rolling offensive/defensive EPA over the last 4 games
- Matchup difficulty (opponent defensive EPA allowed)

If `SEASON` / `WEEK` are `None`, we auto-detect the most recent projection file.


In [13]:
weeks = available_weeks()
if not weeks:
    raise RuntimeError(f"No projection files found in {PROJ_DIR}. Run predict_fantasy.ipynb first.")

if SEASON is None or WEEK is None:
    SEASON, WEEK = weeks[0]

print(f"Using projections: Season {SEASON}, Week {WEEK}")
proj_df = load_projections(SEASON, WEEK)
print(f"  {len(proj_df)} players loaded")
proj_df[["player_display_name","position","team","projected_pts"]].head(10)


Using projections: Season 2025, Week 10
  568 players loaded


,player_display_name,position,team,projected_pts
0,Brock Purdy,QB,SF,24.06
1,Josh Allen,QB,BUF,22.34
2,Bo Nix,QB,DEN,21.05
3,Drake Maye,QB,NE,19.78
4,Jared Goff,QB,DET,18.97
5,Lamar Jackson,QB,BAL,18.97
6,Jalen Hurts,QB,PHI,18.71
7,Kyler Murray,QB,ARI,18.67
8,Trevor Lawrence,QB,JAX,18.63
9,Jordan Love,QB,GB,18.39


## Step 2 — Load DraftKings Salary Data

Set `CSV_PATH` in the Parameters cell to the file you exported from the DK contest
lobby (*Available Players → Export to CSV*). Expected columns:
`Position`, `Name`, `Salary`, `TeamAbbrev`, `AvgPointsPerGame`.

The file is available once DK posts the slate for the week (typically Thursday
for the main Sunday slate, earlier for Thursday Night Football).

**DST rows** use DK's season `AvgPointsPerGame` — no team-defense projection model yet.


In [14]:
dk_df = load_dk_salaries(CSV_PATH)
print(f"DK salary file : {CSV_PATH}")
print(f"  {len(dk_df)} players | positions: {dk_df['position'].value_counts().to_dict()}")
dk_df.sort_values("salary", ascending=False).head(10)


Fetching DK salaries from RotoGuru (nfldfs): Season 2025, Week 10 ...


RuntimeError: RotoGuru returned no player data for 2025 week 10. This week may not be indexed yet, or the site format has changed.

## Step 3 — Build the Player Pool

`merge_projections()` fuzzy-matches each DK player name to our projection file and
pulls through per-stat columns needed for DK scoring.

`calc_dk_proj_pts()` converts the half-PPR model output to **DraftKings Classic** points:

| Adjustment | Positions | Detail |
|---|---|---|
| +0.5 pts/reception | WR, TE | Full PPR (DK=1.0) vs half-PPR (our model=0.5) |
| +0.5 × est. receptions | RB | Rec yards ÷ 7 yds/rec estimate |
| +3 × P(300+ pass yds) | QB | Milestone bonus, Normal-approximated |
| +3 × P(100+ rush yds) | RB | Milestone bonus, Normal-approximated |
| +3 × P(100+ rec yds) | WR, TE | Milestone bonus, Normal-approximated |

The **value** column (`dfs_proj_pts / salary_in_$k`) ranks players by DK efficiency.

**Match quality:**
- `model` — matched to our XGBoost projection ✅
- `dk_avg` — no fuzzy match; using DK season average ⚠️
- `dst` — team defense; always uses DK avg


In [ ]:
players = merge_projections(dk_df, proj_df)
players["dfs_proj_pts"] = calc_dk_proj_pts(players)
players["value"] = (players["dfs_proj_pts"] / (players["salary"] / 1000)).round(2)

unmatched = players[(players["match"] == "dk_avg") & (players["position"] != "DST")]
if len(unmatched):
    print(f"⚠  {len(unmatched)} non-DST players not matched to our model (using DK avg):")
    print(unmatched[["position","name","team","salary","avg_pts"]].to_string(index=False))
else:
    print("All skill-position players matched to model projections ✅")

# Spot-check: DK pts vs raw half-PPR for top players
sample = players[players["match"] == "model"].nlargest(5, "dfs_proj_pts")[
    ["position","name","proj_pts","dfs_proj_pts"]
].copy()
sample["dk_uplift"] = (sample["dfs_proj_pts"] - sample["proj_pts"]).round(2)
print("\nDK scoring uplift — top 5 players:")
print(sample.to_string(index=False))


In [ ]:
# Full player pool — sorted by DK value within each position
pool = (
    players.sort_values(["position","value"], ascending=[True, False])
    [["position","name","team","salary","proj_pts","dfs_proj_pts","value","match"]]
)
print(f"Total player pool: {len(pool)} players")
pool


### Player Pool Analysis

Before optimising it's useful to review the landscape:
- Which positions are salary-compressed (everyone priced similarly)?
- Are there clear value outliers — players priced low relative to projection?
- Does the unmatched list include any must-starts worth manually adjusting?

The position breakdown and top-value lists below answer these quickly.


In [ ]:
print("=== Position breakdown ===")
print(players.groupby("position")[["salary","proj_pts","value"]].agg(["mean","max","min"]).round(1).to_string())

print("\n=== Top 5 by value at each position ===")
for pos in ["QB","RB","WR","TE","DST"]:
    sub = players[players["position"]==pos].nlargest(5,"value")[["name","team","salary","proj_pts","value"]]
    print(f"\n{pos}")
    print(sub.to_string(index=False))


## Step 4 — Optimize the Lineup

The ILP solver maximises total `dfs_proj_pts` (DK Classic full-PPR + milestone bonuses)
subject to the DK Classic roster rules and salary cap. The FLEX slot is filled
automatically — you don't specify whether it should be a RB, WR, or TE.

Modify `LOCKED` / `EXCLUDED` in the Parameters cell and re-run this cell to explore
alternative builds.


In [ ]:
print(f"Running optimizer  budget=${BUDGET:,}  locked={LOCKED}  excluded={EXCLUDED} ...")
lineup = optimize_lineup(players, budget=BUDGET, locked=LOCKED, excluded=EXCLUDED)

if lineup is None:
    print("\n❌  No feasible lineup found. Possible causes:")
    print("   - Not enough players at a position in the salary CSV")
    print("   - Lock constraints are contradictory or exceed the cap")
    print("   - Exclusions remove too many players at a position")
else:
    print("Optimal lineup found ✅")


## Results

The table below shows the optimal 9-player roster with DK slot labels.
`FLEX` indicates the extra skill-position player chosen by the solver.

Review:
- **Total salary** — ideally within $500 of the cap (unused cap = lost points).
- **Salary distribution** — are you paying up at the right positions?
- **Match quality** — any FLEX or anchor picks sourced from `dk_avg`? If so, check the
  actual player's status before locking the lineup.


In [ ]:
if lineup is not None:
    total_sal  = lineup["salary"].sum()
    total_pts  = lineup["dfs_proj_pts"].sum()
    remaining  = BUDGET - total_sal

    print(f"Projected DK pts : {total_pts:.1f}  (half-PPR base: {lineup['proj_pts'].sum():.1f})")
    print(f"Total salary     : ${total_sal:,.0f}")
    print(f"Remaining cap    : ${remaining:,.0f}")
    print()

    display_cols = ["Slot","name","team","salary","proj_pts","dfs_proj_pts","value","match"]
    print(lineup[display_cols].to_string(index=False))


In [ ]:
# Salary allocation by position
if lineup is not None:
    sal_by_pos = lineup.groupby("position")["salary"].sum().sort_values(ascending=False)
    pct        = (sal_by_pos / total_sal * 100).round(1)
    print("=== Salary allocation ===")
    for pos, sal in sal_by_pos.items():
        print(f"  {pos:<4}  ${sal:>6,.0f}  ({pct[pos]}%)")
    print(f"  {'TOTAL':<4}  ${total_sal:>6,.0f}")


In [ ]:
# Export lineup in DK upload format
if lineup is not None:
    from collections import defaultdict

    # DraftKings Classic CSV import expects exactly these 9 columns, in order.
    # Duplicate headers (RB, RB / WR, WR, WR) are valid in the DK template.
    slot_order = ["QB", "RB", "RB", "WR", "WR", "WR", "TE", "FLEX", "DST"]

    # _assign_slots (optimizer) labels each row QB/RB/WR/TE/FLEX/DST with the
    # counts that match slot_order: 1 QB, 2 RB, 3 WR, 1 TE, 1 FLEX, 1 DST.
    by_slot = defaultdict(list)
    for _, r in lineup.iterrows():
        by_slot[r["Slot"]].append(r["name"])

    # Fill each slot_order position, consuming names from the matching label.
    names_in_order, consumed = [], defaultdict(int)
    for slot in slot_order:
        bucket = by_slot.get(slot, [])
        idx = consumed[slot]
        names_in_order.append(bucket[idx] if idx < len(bucket) else "")
        consumed[slot] += 1

    dk_upload = pd.DataFrame([names_in_order], columns=slot_order)

    out_path = Path(f"dk_lineup_{SEASON}_week{WEEK:02d}.csv")
    dk_upload.to_csv(out_path, index=False)
    print(f"Lineup saved to {out_path}")
    print("Upload to DraftKings: My Lineups → Import Lineups → Upload CSV")
    print(dk_upload.to_string(index=False))


## Next Steps & Future Improvements

### Immediate
- **Review unmatched players** — any `dk_avg` players in your lineup warrant a manual
  projection check (FantasyPros consensus, snap count trends, etc.).
- **Check injury reports** — run after the official Thursday injury report drops; re-run
  the optimizer with newly-ruled-out players added to `EXCLUDED`.

### Model improvements
1. **DST projection model** — train on defensive EPA allowed, implied team total, home/away,
   and surface. Replace `dk_avg` fallback for DST.
2. **Multi-lineup GPP optimizer** — generate N lineups with ownership diversity constraints,
   forcing variation in the FLEX pick and at least one different anchor per lineup.
3. **Game-stacking correlation** — prefer combinations from the same game (e.g. QB + WR1 +
   opponent WR) which correlate positively in high-scoring contests.
4. **Salary movement signal** — compare `salary` to prior-week salary; big drops may indicate
   recency information the season-average doesn't yet capture.
5. **Integration with predict_fantasy.ipynb** — run this pipeline immediately after the
   weekly projections are generated so the process is one command end-to-end.
